In [ ]:
# Installation for Azure Event Hub client library (run once)
# Event Hub supports Kafka protocol, so we use kafka-python for compatibility
!pip install kafka-python

In [ ]:
# Imports and basic configuration
import json
import uuid
import random
import time
from datetime import datetime, timedelta, timezone
from kafka import KafkaProducer
import numpy as np

# Configurable parameters
EVENTS_PER_SECOND = 10  # Rate of events per second
SIMULATION_HOURS = 8   # Duration to run the simulation in hours

In [ ]:
# Azure Event Hub configuration parameters
# Azure Event Hub is Kafka-compatible, allowing us to use the Kafka protocol

# EVENT_HUB_NAMESPACE: Your Event Hub namespace (format: <namespace>.servicebus.windows.net:9093)
EVENT_HUB_NAMESPACE = 'TBD'

# EVENT_HUB_NAME: The name of your Event Hub (topic in Kafka terminology)
EVENT_HUB_NAME = 'TBD'

# EVENT_HUB_CONNECTION_STRING: Get this from Azure Portal > Event Hubs > Shared access policies
EVENT_HUB_CONNECTION_STRING = 'TBD'

# Authentication configuration (required for Azure Event Hub)
SASL_USERNAME = '$ConnectionString'  # This value should remain as-is for Event Hub
SASL_PASSWORD = EVENT_HUB_CONNECTION_STRING
SASL_MECHANISM = 'PLAIN'
SECURITY_PROTOCOL = 'SASL_SSL'

In [ ]:
producer = None

def init_event_hub_producer(
    namespace, 
    sasl_username, 
    sasl_password, 
    sasl_mechanism='PLAIN', 
    security_protocol='SASL_SSL'
):
    """
    Initialize producer for Azure Event Hub using Kafka protocol.
    
    Args:
        namespace: Event Hub namespace with port (e.g., 'namespace.servicebus.windows.net:9093')
        sasl_username: Authentication username (use '$ConnectionString' for Event Hub)
        sasl_password: Event Hub connection string
        sasl_mechanism: SASL mechanism (default: 'PLAIN')
        security_protocol: Security protocol (default: 'SASL_SSL')
    """
    global producer
    kafka_config = {
        'bootstrap_servers': [namespace],
        'value_serializer': lambda v: json.dumps(v).encode('utf-8'),
        'acks': 'all',
        'retries': 3,
        'security_protocol': security_protocol,
        'sasl_mechanism': sasl_mechanism,
        'sasl_plain_username': sasl_username,
        'sasl_plain_password': sasl_password,
        'api_version_auto_timeout_ms': 10000,
        'request_timeout_ms': 30000,
        'connections_max_idle_ms': 540000
    }
    
    producer = KafkaProducer(**kafka_config)
    print(f"Azure Event Hub producer initialized to namespace: {namespace}")

def send_event_to_event_hub(event, event_hub_name):
    """
    Send event to Azure Event Hub.
    
    Args:
        event: Event data to send (will be JSON serialized)
        event_hub_name: Name of the Event Hub to send to
    """
    if producer is None:
        raise Exception("Event Hub producer not initialized")
    try:
        future = producer.send(event_hub_name, event)
        record_metadata = future.get(timeout=10)
        print(f"Event sent to Event Hub '{record_metadata.topic}' "
              f"partition {record_metadata.partition} offset {record_metadata.offset}")
        producer.flush()
    except Exception as e:
        print(f"Error sending event to Event Hub: {e}")

In [ ]:
from datetime import datetime
import time
import random

def generateLoRaWanElsysErs1Event():
    event = {
        "end_device_ids": {
            "device_id": "eui-a81758fffe08a90f",
            "application_ids": {
                "application_id": "svelde-elsys-ers"
            }
        },
        "received_at" : datetime.now().isoformat() + "Z",
        "uplink_message" : {
            "decoded_payload" : {
                "humidity": str(random.randint(60, 70)), # 64,
                "light": str(random.randint(2200, 2500)), # 2417,
                "temperature": str(random.randint(170, 190) / 10), # 18.5,
                "vdd": str(random.randint(3400, 3450)) # 3415                
            }
        }
    }
    
    return event

def generateLoRaWanElsysErs2Event():
    event = {
        "end_device_ids": {
            "device_id": "eui-a81758fffe08a8e7",
            "application_ids": {
                "application_id": "svelde-elsys-ers"
            }
        },
        "received_at" : datetime.now().isoformat() + "Z",
        "uplink_message" : {
            "decoded_payload" : {
                "humidity": str(random.randint(60, 70)), # 64,
                "light": str(random.randint(2200, 2500)), # 2417,
                "temperature": str(random.randint(170, 190) / 10), # 18.5,
                "vdd": str(random.randint(3400, 3450)) # 3415                
            }
        }
    }
    
    return event

def generateLoRaWanGenericNode1Event():
    event = {
        "end_device_ids": {
            "device_id": "eui-0004a310001ab228",
            "application_ids": {
                "application_id": "svelde-ttn-generic-node"
            }
        },
        "received_at" : datetime.now().isoformat() + "Z",
        "uplink_message" : {
            "decoded_payload" : {
                "batt_volt": str(random.randint(32, 36) / 10), # 3.4,
                "button": str(random.randint(0, 1)), # 0 | 1,
                "temperature": str(random.randint(180, 200) / 10), # 19.5,
                "humidity": str(random.randint(500, 510) / 10) # 50.7              
            }
        }
    }
    
    return event    


def generateLoRaWanCardTrackerEvent():

    now = datetime.now()

    epoch = int(time.mktime(now.timetuple())) * 1000 + int(now.microsecond / 1000), # 1759579139000,

    event = {
        "end_device_ids": {
            "device_id": "eui-2cf7f1c053200054",
            "application_ids": {
                "application_id": "svelde-sensecap-t1000-card-tracker"
            }
        },
        "received_at" : now.isoformat() + "Z",
        "uplink_message" : {
            "decoded_payload" : {
                "err": 0,
                "messages": [
                    [
                    {
                        "measurementId": "4200",
                        "measurementValue": [],
                        "motionId": 0,
                        "timestamp": epoch,
                        "type": "Event Status"
                    },
                    {
                        "measurementId": "4197",
                        "measurementValue": 5.61234,
                        "motionId": 0,
                        "timestamp": epoch,
                        "type": "Longitude"
                    },
                    {
                        "measurementId": "4198",
                        "measurementValue": 51.461234,
                        "motionId": 0,
                        "timestamp": epoch,
                        "type": "Latitude"
                    },
                    {
                        "measurementId": "3000",
                        "measurementValue": random.randint(50, 51),
                        "motionId": 0,
                        "timestamp":  epoch,
                        "type": "Battery"
                    }
                    ]
                ],
                "payload": "090000000068e10c030055be96031138a033",
                "valid": "true"
            }
        }
    }
    
    return event    


In [ ]:
def generateEvents(
    event_hub_name,
    send_func=None, 
    print_events=True
):
    """
    Generate and send simulated LoRaWAN device events.
    
    Args:
        event_hub_name: Name of the Event Hub to send events to
        send_func: Function to call to send events (optional)
        print_events: Whether to print events to console (default: True)
    """
    try:
        while True:
            # Generate and send card tracker event
            cardTrackerEvent = generateLoRaWanCardTrackerEvent()    
            if print_events:
                print(json.dumps(cardTrackerEvent))
        
            if send_func:
                send_func(cardTrackerEvent, event_hub_name)

            time.sleep(10)

            # Generate and send Elsys ERS sensor 1 event
            loRaWanElsysErs1Event = generateLoRaWanElsysErs1Event()    
            if print_events:
                print(json.dumps(loRaWanElsysErs1Event))
        
            if send_func:
                send_func(loRaWanElsysErs1Event, event_hub_name)

            time.sleep(10)

            # Generate and send Elsys ERS sensor 2 event
            loRaWanElsysErs2Event = generateLoRaWanElsysErs2Event()    
            if print_events:
                print(json.dumps(loRaWanElsysErs2Event))
        
            if send_func:
                send_func(loRaWanElsysErs2Event, event_hub_name)

            time.sleep(10)

            # Generate and send generic node event
            genericNode1Event = generateLoRaWanGenericNode1Event()    
            if print_events:
                print(json.dumps(genericNode1Event))
        
            if send_func:
                send_func(genericNode1Event, event_hub_name)

            time.sleep(10)
    except KeyboardInterrupt:
        if producer:
            producer.close()
        print("\nEvent generation stopped.")

In [ ]:
# Initialize Azure Event Hub Producer with authentication
init_event_hub_producer(
    EVENT_HUB_NAMESPACE, 
    SASL_USERNAME, 
    SASL_PASSWORD,
    SASL_MECHANISM,
    SECURITY_PROTOCOL
)

print(f"Starting LoRaWAN event generation at {datetime.now()}")
print(f"Sending to Event Hub: {EVENT_HUB_NAME}")
print("Press Ctrl+C to stop\n")

# Start generating and sending LoRaWAN events to Azure Event Hub
generateEvents(
    event_hub_name=EVENT_HUB_NAME,
    send_func=send_event_to_event_hub,
    print_events=True
)

print(f"Event generation completed at {datetime.now()}")